<a href="https://colab.research.google.com/github/prasa129/Econometrics/blob/main/NN_OLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Neural Network OLS

11-25-2025

A short demo of how a very simple neural network can exactly replicate ordinary least squares (OLS) linear regression. I fit an OLS model to a dataset split into train and test, then I fit a neural network with the same inputs and target.

The architechture is a single linear layer with one output and no hidden units or nonlinear activation, thus representing linear functions of the predictors. I minimize MSE using gradient descent.

Because the OLS problem is convex and the network class coincides with the class of linear models, the trained network converges to OLS solution, up to numerical error. To compare inference measures ($p$-values, CIs, $t$-statistics, etc.), I use empirical bootstrap for the NN.

In [1]:
# imports
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy import stats
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense

# set global display precision
pd.set_option("display.float_format", "{:.4f}".format)

# load boston housing data from url
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"

# read csv
df = pd.read_csv(url)

# info
print("\n=== Datatypes and Counts ===")
print(df.info())

# summary statistics
print("\n=== Summary Statistics ===")
print(df.describe())

# extract target column
y = df["medv"].values.astype(float)

# extract feature matrix
X = df.drop(columns=["medv"]).values.astype(float)

# store feature names
feature_names = df.drop(columns=["medv"]).columns.tolist()

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.30,
                                                    random_state=1776)

# create scaler
scaler = StandardScaler()

# standardize training features
X_train_scaled = scaler.fit_transform(X_train)

# standardize test features
X_test_scaled = scaler.transform(X_test)

# basic diagnostics
print("\n=== Diagnostics ===")
print("X_train shape:", X_train_scaled.shape)
print("X_test shape: ", X_test_scaled.shape)
print("Features:", feature_names)



=== Datatypes and Counts ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     506 non-null    float64
 1   zn       506 non-null    float64
 2   indus    506 non-null    float64
 3   chas     506 non-null    int64  
 4   nox      506 non-null    float64
 5   rm       506 non-null    float64
 6   age      506 non-null    float64
 7   dis      506 non-null    float64
 8   rad      506 non-null    int64  
 9   tax      506 non-null    int64  
 10  ptratio  506 non-null    float64
 11  b        506 non-null    float64
 12  lstat    506 non-null    float64
 13  medv     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB
None

=== Summary Statistics ===
          crim       zn    indus     chas      nox       rm      age      dis  \
count 506.0000 506.0000 506.0000 506.0000 506.0000 506.0000 506.0000 506.0000   
mean    3.

The data is suitable. Columns are numeric, there are no missing values, and the number of observations considerably exceeds the number of features. A 70/30 train/test split leaves enough observations to negate power concerns and give meaningful test error measures. I fit OLS from scratch.

In [2]:
def ols_inference(X, y):
    """
    Closed-form OLS with analytic SE, t-stats, p-values, and 95% CIs.
    Args:
    -----
        X (numpy.ndarray): Feature matrix (n, p).
        y (numpy.ndarray): Target vector (n,1).
    Returns:
    --------
        results (dict): Contains OLS coefficients, intercept, SEs, t-stats,
                        p-values, and CIs.
    Fit model y = beta_0 + X beta + e.
    """

    # sample size and number of features
    n, p = X.shape

    # add intercept column to design matrix
    X_design = np.column_stack((np.ones(n), X))

    # compute X'X
    XtX = X_design.T @ X_design

    # invert X'X
    XtX_inv = np.linalg.inv(XtX)

    # OLS coefficients: beta_hat = (X'X)^(-1)X'y
    beta_hat = XtX_inv @ (X_design.T @ y)

    # intercept estimate
    intercept_hat = beta_hat[0]

    # slope coefficients
    coef_hat = beta_hat[1:]

    # fitted values
    y_hat = X_design @ beta_hat

    # residuals
    residuals = y - y_hat

    # degrees of freedom
    df = n - p - 1

    # residual variance estimate
    sigma2_hat = (residuals @ residuals) / df

    # variance-covariance matrix of beta
    var_beta = sigma2_hat * XtX_inv

    # standard errors of all parameters
    se_beta = np.sqrt(np.diag(var_beta))

    # standard errors for slope coefficients
    se_coef = se_beta[1:]

    # t-statistics for coefficients
    t_stats = coef_hat / se_coef

    # two sided p-values
    p_values = 2.0 * (1.0 - stats.t.cdf(np.abs(t_stats), df=df))

    # 97.5% quantile for 95% CI
    t_crit = stats.t.ppf(0.975, df=df)

    # lower CI bound
    ci_lower = coef_hat - t_crit * se_coef

    # upper CI bound
    ci_upper = coef_hat + t_crit * se_coef

    # TSS
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    # RSS
    ss_res = np.sum(residuals ** 2)

    # R2
    r2 = 1.0 - ss_res / ss_tot

    # dict results
    results = {"coef": coef_hat, "intercept": intercept_hat, "se": se_coef,
               "t": t_stats, "p": p_values,
               "ci_lower": ci_lower, "ci_upper": ci_upper,
               "r2": r2, "df": df}
    return results


# compute analytic OLS stats on training set
ols_stats = ols_inference(X_train_scaled, y_train)

The neural network has the form

$$
\hat{y} = f_\theta(x) = w^\top x + b,
$$

where $x \in \mathbb{R}^p$ is the feature vector, $w \in \mathbb{R}^p$ is the weight vector, and $b \in \mathbb{R}$ is a bias (intercept) term. There are no hidden layers and no nonlinear activation functions, thus the model is exactly the same functional form as a linear regression. The training objective is the empirical MSE,

$$
L(w,b) = \frac{1}{n} \sum_{i=1}^n \bigl(y_i - (w^\top x_i + b)\bigr)^2,
$$

which is the same loss minimized by OLS (up to a constant scaling). This objective is a quadratic, strictly convex function of $(w,b)$, so it has a unique global minimizer.

OLS finds this minimizer in closed form via the normal equations. The neural network finds the same minimizer by iterative gradient-based optimization. As long as the optimizer uses a reasonable learning rate and is run for enough iterations, gradient descent on this convex loss converges to the global optimum. Since the function class (all affine maps $x \mapsto w^\top x + b$) and the loss function (MSE) match exactly, the optimal network parameters $(\hat{w}, \hat{b})$ coincide with the OLS estimates $(\hat{\beta}, \hat{\beta}_0)$ up to numerical tolerance. Thus I expect that $R^2$ values and the coefficient vectors from the neural network and OLS are essentially identical.

In [3]:
def build_linear_nn(n_features, learning_rate=0.1):
    """
    Build a 1-layer linear NN: y = X w + b.
    Args:
    -----
        n_features (int)     : Number of input features.
        learning_rate (float): Learning rate for SGD optimizer.
    Returns:
    --------
        model (tf.keras.Model): Compiled linear regression NN.
    """

    # create sequential model
    model = Sequential()

    # add explicit Input layer
    model.add(Input(shape=(n_features,)))

    # add linear Dense layer
    model.add(Dense(units=1, activation=None, use_bias=True))

    # use SGD optimizer with passed in learning rate
    opt = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    # compile model with MSE loss
    model.compile(optimizer=opt, loss="mse")

    return model

# set seed for reproducability
tf.random.set_seed(1776)

# build NN for full-sample training
nn_model = build_linear_nn(n_features=X_train_scaled.shape[1],learning_rate=0.1)

# train using full-batch gradient descent
history = nn_model.fit(X_train_scaled, y_train, batch_size=len(X_train_scaled),
                       epochs=2000, verbose=0)

# extract NN weights from last (Dense) layer
w, b = nn_model.layers[-1].get_weights()

# flatten weight vector
nn_coef = w.ravel()

# convert intercept from array to scalar
nn_intercept = b.item()

# compute L2 difference between OLS and NN coefficients
print("\nL2 norm of (OLS coef - NN coef):", coef_diff)
coef_diff = np.linalg.norm(ols_stats["coef"] - nn_coef)
print(coef_diff)

# compute OLS, NN fitted values on train
y_hat_ols_train = X_train_scaled @ ols_stats["coef"] + ols_stats["intercept"]
y_hat_nn_train = nn_model.predict(X_train_scaled, verbose=0).ravel()

# compute OLS, NN fitted values on test
y_hat_ols_test = X_test_scaled @ ols_stats["coef"] + ols_stats["intercept"]
y_hat_nn_test = nn_model.predict(X_test_scaled, verbose=0).ravel()

# compute R2 for OLS, NN on train, test
r2_ols_train = r2_score(y_train, y_hat_ols_train)
r2_nn_train = r2_score(y_train, y_hat_nn_train)
r2_ols_test = r2_score(y_test, y_hat_ols_test)
r2_nn_test = r2_score(y_test, y_hat_nn_test)

# print R^2 comparison
print("\n=== R^2 comparison ===")
print(f"Train R2 (OLS analytic): {r2_ols_train:.4f}")
print(f"Train R2 (NN)          : {r2_nn_train:.4f}")
print(f"Test  R2 (OLS analytic): {r2_ols_test:.4f}")
print(f"Test  R2 (NN)          : {r2_nn_test:.4f}")


L2 norm of (OLS coef - NN coef): 1.367045141676349e-05

=== R^2 comparison ===
Train R2 (OLS analytic): 0.7324
Train R2 (NN)          : 0.7324
Test  R2 (OLS analytic): 0.7122
Test  R2 (NN)          : 0.7122


The L2 norm of the difference between the OLS and NN coefficient estimate vectors is near zero, and the train/test $R^2$ measures are the same. The fits match.

OLS gives inference measures (e.g. SE, $p$-values, CIs) based on statistical theory. A neural network outputs only coefficient estimates. Thus to obtain inference measures for the neural network, coefficients, I use empirical bootstrap.

The procedure is:

(i) draw $B$ bootstrap samples from the training data by sampling observations with replacement

(ii) for each bootstrap sample, retrain the same single-layer neural network to convergence and record the estimated weight vector

(iii) use the resulting $B$ sets of coefficients to approximate the sampling distribution of the network parameters.

The sample standard deviation of the bootstrap coefficients provides an estimate of the standard error, percentile intervals give nonparametric confidence intervals, and simple ratios (mean divided by standard error) yield bootstrap $t$-statistics and corresponding $p$-values.

I expect bootstrap based quantities line up closely with the classical OLS analytic formulas. The procedure is time consuming.

In [4]:
def bootstrap_nn_coefficients(X,y,n_boot=80,epochs=400,
                              learning_rate=0.1,random_state=0):
    """
    Empirical bootstrap for NN-based linear regression coefficients.
    Args:
    -----
        X (numpy.ndarray): Feature matrix (n, p).
        y (numpy.ndarray): Target vector (n,).
        n_boot (int)     : Number of bootstrap replications.
        epochs (int)     : Epochs per bootstrap fit.
        learning_rate    : Learning rate for SGD.
        random_state (int): Seed for bootstrap resampling.
    Returns:
    --------
        results (dict): Contains bootstrap means, SEs, t-stats, p-values,
                        and CIs.
    For each bootstrap:
      1) Resample (X, y) with replacement.
      2) Fit a fresh 1-layer linear NN.
      3) Store estimated coefficients.
    """

    # sample size and number of features
    n, p = X.shape

    # random number generator
    rng = np.random.default_rng(random_state)

    # storage for bootstrap coefficients
    boot_coefs = np.zeros((n_boot, p))

    # storage for bootstrap intercepts
    boot_intercepts = np.zeros(n_boot)

    # loop over bootstrap replications
    for b in range(n_boot):

        # resample indices with replacement
        idx = rng.integers(0, n, size=n)

        # bootstrap features
        Xb = X[idx]

        # bootstrap targets
        yb = y[idx]

        # set seed
        tf.random.set_seed(b)

        # build new linear NN
        model = build_linear_nn(n_features=p, learning_rate=learning_rate)

        # fit NN with full-batch training
        model.fit(Xb, yb, batch_size=len(Xb), epochs=epochs, verbose=0)

        # extract weights from last (Dense) layer
        w_b, b_b = model.layers[-1].get_weights()

        # store coefficients
        boot_coefs[b, :] = w_b.ravel()

        # store intercept
        boot_intercepts[b] = b_b.item()

    # bootstrap mean of coefficients
    coef_mean = np.mean(boot_coefs, axis=0)

    # bootstrap standard errors
    coef_se = np.std(boot_coefs, axis=0, ddof=1)

    # bootstrap t-statistics
    t_stats = coef_mean / coef_se

    # empirical two-sided p-values
    p_values = np.mean(np.abs(boot_coefs - coef_mean[None, :]) >= np.abs(coef_mean[None, :]), axis=0)

    # p2.5 for lower CI
    ci_lower = np.percentile(boot_coefs, 2.5, axis=0)

    # p97.5 for upper CI
    ci_upper = np.percentile(boot_coefs, 97.5, axis=0)

    # dict results
    results = {"coef_mean": coef_mean, "coef_se": coef_se,
               "t": t_stats, "p": p_values,
               "ci_lower": ci_lower,"ci_upper": ci_upper,
               "boot_coefs": boot_coefs,"boot_intercepts": boot_intercepts}
    return results


# run bootstrap inference for NN
nn_boot = bootstrap_nn_coefficients(X_train_scaled, y_train,
                                    n_boot=80, epochs=400,
                                    learning_rate=0.1, random_state=123)


# build comparison DataFrame
compare_df = pd.DataFrame({"coef_ols": ols_stats["coef"],
                           "se_ols": ols_stats["se"],
                           "t_ols": ols_stats["t"],
                           "p_ols": ols_stats["p"],
                           "ci_ols_lo": ols_stats["ci_lower"],
                           "ci_ols_hi": ols_stats["ci_upper"],
                           "coef_nn_boot_mean": nn_boot["coef_mean"],
                           "se_nn_boot": nn_boot["coef_se"],
                           "t_nn_boot": nn_boot["t"],
                           "p_nn_boot": nn_boot["p"],
                           "ci_nn_boot_lo": nn_boot["ci_lower"],
                           "ci_nn_boot_hi": nn_boot["ci_upper"]},
                          index=feature_names)

# check results
print("\n=== OLS analytic vs NN bootstrap inference ===")
display(compare_df.round(4))



=== OLS analytic vs NN bootstrap inference ===


,coef_ols,se_ols,t_ols,p_ols,ci_ols_lo,ci_ols_hi,coef_nn_boot_mean,se_nn_boot,t_nn_boot,p_nn_boot,ci_nn_boot_lo,ci_nn_boot_hi
crim,-0.5267,0.3697,-1.4244,0.1552,-1.2539,0.2006,-0.5266,0.3687,-1.4282,0.1875,-1.1312,0.1473
zn,1.2029,0.3707,3.2447,0.0013,0.4737,1.9321,1.1666,0.4182,2.7900,0.0000,0.2385,1.9502
indus,0.1097,0.5329,0.2059,0.8370,-0.9385,1.1580,0.0093,0.3770,0.0247,0.9875,-0.6948,0.7829
chas,1.0427,0.2556,4.0788,0.0001,0.5399,1.5455,1.0478,0.3991,2.6256,0.0125,0.3016,1.7564
nox,-2.4561,0.5188,-4.7340,0.0000,-3.4766,-1.4356,-2.3712,0.5990,-3.9588,0.0000,-3.4764,-1.3778
rm,1.8370,0.3377,5.4392,0.0000,1.1727,2.5014,1.8899,0.6339,2.9813,0.0125,0.9552,3.1430
age,-0.0174,0.4408,-0.0395,0.9685,-0.8844,0.8496,0.0173,0.5270,0.0328,0.9625,-1.0729,0.9092
dis,-3.1557,0.4879,-6.4675,0.0000,-4.1155,-2.1960,-3.1170,0.5675,-5.4924,0.0000,-4.0894,-2.1407
rad,2.6238,0.6748,3.8883,0.0001,1.2965,3.9510,2.5484,0.6138,4.1520,0.0000,1.4545,3.7485
tax,-2.0346,0.7432,-2.7375,0.0065,-3.4964,-0.5727,-1.9558,0.5558,-3.5186,0.0000,-2.9741,-1.0727


The results show that a single-layer linear neural network, trained with MSE loss, reproduces the OLS solution almost exactly. The $R^2$ on both train and test sets match, the coefficient vectors are the same, and the bootstrap-based uncertainty measures for the network coefficients are reasonably close to the analytic OLS standard errors, $t$-statistics, $p$-values, and confidence intervals given very few bootstraps ($B=80$).

With a simple enough architecture and loss, a NN can reduce to a familiar statistical model. Standard inferential tools (e.g. bootstrap) can be layered on top of NN training to recover regression style uncertainty quantification, bridging the gap between econometrics and modern ML in a tangible, transparent way.